# Testing YOLO on Single Image from Simulator
### Starting image
![simulator gate](start.png)

In [6]:
# check environment

import sys

print(sys.executable)

c:\Users\annie\.pyenv\pyenv-win\versions\3.14.2\python.exe


In [6]:
from ultralytics import YOLO
# Load a pretrained YOLO26 nano model
model = YOLO("yolo26n.pt")
# Run inference on an image
results = model("start.png")
# Display results
results[0].show()


image 1/1 c:\Users\annie\PersonalProjects\dabjak4\ai-grand-prix\start.png: 384x640 1 train, 71.7ms
Speed: 3.1ms preprocess, 71.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)


Results with no training are shown below:

![no training yolo26 detection](yolo-detection-no-training.png)

Train: predicted class

0.25: confidence score

# Training with Online Dataset

## Download the Dataset

Note: the following code just downloads the dataset, so once you have it, you can just comment it out if you don't want it to run again. If it does run again, it won't do anything. If the install was incomplete, delete the folder and rerun the code.

In [7]:
import os
from roboflow import Roboflow
from dotenv import load_dotenv

load_dotenv()

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
rf = Roboflow(ROBOFLOW_API_KEY)
project = rf.workspace("drone-racing").project("airsim-drone-racing-lab-gates")
version = project.version(2)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...


Dataset includes training set, validation set, and test set. Use the training set for learning, the validation set to check on training, and the test set at the end of training.

The labels are basically the boxes that were manually traced around the images from the dataset (annotations included as part of it). They each have info in the form "0 0.421875 0.496875 0.41640625 0.84453125". The first number is the corresponding image number. The next 2 numbers are the coordinates of the top left corner and the last 2 numbers are the coordinates of the bottom right corner.

Also make sure that the paths in data.yaml for test, train, and valid are correct.

## Training

Note: there's a lot of parameters for model training (see docs in Resources), so I only used what I expect to make the largest impact. If it needs refining, we could look at others. 

In [ ]:
trained_model = model.train(
    data="AirSim-Drone-Racing-Lab-Gates-2/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=100,
    seed=0
)

Ultralytics 8.4.90  Python-3.12.6 torch-2.13.0+cpu CPU (13th Gen Intel Core i7-13700H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=AirSim-Drone-Racing-Lab-Gates-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [15]:
trained_model = YOLO("runs/detect/train-2/weights/best.pt")
results = trained_model("AirSim-Drone-Racing-Lab-Gates-2/train/images/1_png.rf.9a7bf6287105dcc23ce61570751688b2.jpg", conf=0.05)
print(len(results[0].boxes), "detections")
results[0].show()


image 1/1 c:\Users\annie\PersonalProjects\dabjak4\ai-grand-prix\AirSim-Drone-Racing-Lab-Gates-2\train\images\1_png.rf.9a7bf6287105dcc23ce61570751688b2.jpg: 640x640 2 gates, 73.2ms
Speed: 1.9ms preprocess, 73.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
2 detections


* patience (default 100): number of epochs with no validation improvement before early stopping
* batch (default 16): batch size
* epochs (default 100): number of epochs
* imgsz (default 640 px): image size
* seed (default 0): sets the seed

### Considerations for Model Improvement

1. We can try changing the imgsz value (see issue #2 in the repo). I think this could potentially matter a lot for VQ2.
2. We can change the learning rate to see if that makes a difference, if improvements to the model are needed.
3. We can technically choose to not use the pretrained model, although this likely won't result in any improvements.

### Considerations for Faster Model Learning

1. Using the workers parameter sets the number of threads to run at once. The default is 8, so if your GPU can handle it, try setting it higher.
2. Setting cache to True to allow caching. The default is False.

## Training Results

| Base Model | Model Name | Parameters | Confidence Interval for start.png | Time to Process start.png | Training Time | Notes |
| - | - | - | - | - | - | - |
| Normal YOLO26 | N1 | data="AirSim-Drone-Racing-Lab-Gates-2/data.yaml", epochs=100, imgsz=640, batch=16, patience=100, seed=0 | N/A | doesn't matter | 2+ hours | Doesn't detect gates because the training dataset envrionment is too different from the simulator environment. |

# Resources

## Yolo v26 (and Related)

YOLO v26 usage docs: https://docs.ultralytics.com/models/yolo26#usage-examples

YOLO v26 training docs (e.g. train parameters): https://docs.ultralytics.com/modes/train#musgd-optimizer

Roboflow training dataset: https://universe.roboflow.com/drone-racing/airsim-drone-racing-lab-gates/dataset/2

## About Machine Learning

Epochs and batches: https://www.geeksforgeeks.org/machine-learning/epoch-in-machine-learning/